# SSAFY AI Challenge: Recycling VQA — H100 Top-1 Notebook

**Target**: 0.93+ accuracy on recycling image VQA (Korean multiple-choice)

**Model**: Qwen2.5-VL-7B-Instruct with LoRA fine-tuning on H100 80GB

**Key improvements over baseline (0.87)**:
- 672px resolution (3.1x more pixels for counting questions)
- bf16 full precision (no quantization loss)
- Domain-specific Korean recycling prompt
- Label masking (gradient signal focused on answer)
- Logprob inference (no parse errors)
- TTA with answer-choice shuffling (position-bias elimination)
- Filtered dev data with majority vote for extra training rows
- Early stopping + timing/memory probes

## Cell 1: Environment Setup

Install all required packages and verify GPU availability.

# Cell 0: vLLM 환경 사전 설정 (가장 먼저 실행)

In [ ]:
# ============================================================
# 반드시 다른 모든 셀보다 먼저 실행하세요!
# vLLM multiprocessing 호환성 + CUDA 사전 설정
# ============================================================
import os
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ATTENTION_BACKEND'] = 'FLASH_ATTN'
print('vLLM environment configured.')


In [ ]:
# ============================================================
# 환경 설정 (보수적 + 안전)
# ============================================================

# 0. Colab 기본 PyTorch/CUDA 확인 (절대 재설치하지 않음!)
import torch
print(f'Pre-installed PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')
else:
    raise RuntimeError('CUDA not available! GPU 런타임을 설정하세요.')

# 1. Flash Attention — 사전빌드 휠 자동 탐지
import sys, subprocess
cuda_ver = torch.version.cuda.replace('.', '')  # e.g., '121' or '124'
torch_ver = '.'.join(torch.__version__.split('.')[:2])  # e.g., '2.5'
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'  # e.g., 'cp312'
print(f'\nFlash-attn 환경: CUDA {torch.version.cuda}, PyTorch {torch_ver}, Python {py_ver}')

try:
    import flash_attn
    print(f'Flash-attn already installed: {flash_attn.__version__}')
except ImportError:
    # 사전빌드 휠 시도 (mjun0812 repo)
    wheel_url = (
        f'https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/'
        f'v0.7.16/flash_attn-2.8.3+cu{cuda_ver[:3]}torch{torch_ver}-{py_ver}-{py_ver}-linux_x86_64.whl'
    )
    print(f'Trying prebuild wheel: {wheel_url}')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', wheel_url],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Prebuild wheel not found, building from source (20-30분 소요)...')
        !pip install -q flash-attn --no-build-isolation
    else:
        print('Flash-attn prebuild wheel installed (30초)')

# 2. Core dependencies (torch 제외! Colab 기본 PyTorch를 유지)
!pip install -q \
    "transformers>=4.49.0,<5.0.0" \
    "accelerate>=0.34.2,<1.0.0" \
    "peft>=0.13.2,<1.0.0" \
    "scikit-learn>=1.3.0" \
    "albumentations>=1.3.0,<2.0.0" \
    "pillow" "pandas" "tqdm"

# 3. vLLM (추론 가속, optional — 실패해도 HF fallback 가능)
try:
    import vllm
    print(f'vLLM already installed: {vllm.__version__}')
except ImportError:
    print('Installing vLLM...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'vllm>=0.8.0'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'vLLM install failed (HF fallback will be used): {result.stderr[:200]}')
    else:
        print('vLLM installed successfully')

# 4. 설치 결과 검증
print('\n' + '=' * 50)
print('DEPENDENCY VERIFICATION')
print('=' * 50)

deps = {
    'torch': lambda: torch.__version__,
    'transformers': lambda: __import__('transformers').__version__,
    'peft': lambda: __import__('peft').__version__,
    'accelerate': lambda: __import__('accelerate').__version__,
    'flash_attn': lambda: __import__('flash_attn').__version__,
    'albumentations': lambda: __import__('albumentations').__version__,
    'sklearn': lambda: __import__('sklearn').__version__,
    'vllm': lambda: __import__('vllm').__version__,
}

for name, get_ver in deps.items():
    try:
        ver = get_ver()
        print(f'  {name:20s} {ver:20s} OK')
    except Exception as e:
        status = 'OPTIONAL' if name in ('vllm', 'flash_attn') else 'MISSING'
        print(f'  {name:20s} {"—":20s} {status}')

# GPU 상태 재확인 (torch가 깨지지 않았는지)
assert torch.cuda.is_available(), 'CUDA broke after pip install! 런타임을 재시작하세요.'
print(f'\nGPU OK: {torch.cuda.get_device_name(0)}')


## Cell 2: Imports and Constants

All imports and hyperparameters in one place for easy tuning.

In [ ]:
import os, re, math, random, json, gc, time, itertools
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any, Optional, Tuple
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm

# 보수적 이미지 증강 (밝기/대비/JPEG만, 정답에 영향 없는 변환만)
import albumentations as A
TRAIN_TRANSFORM = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.3),
    A.ImageCompression(quality_lower=75, quality_upper=100, p=0.2),
])

Image.MAX_IMAGE_PIXELS = None
device = "cuda"

# -- Hyperparameters (Model A) --
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
IMAGE_SIZE = 672
SEED_A = 42
BATCH_SIZE = 1
GRAD_ACCUM = 4
NUM_EPOCHS = 3
LR_A = 1e-5
LORA_R_A = 64
LORA_ALPHA_A = 128
LORA_DROPOUT = 0.05
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
DEV_AGREEMENT_THRESHOLD = 3  # out of 5 annotators

# === 전체 데이터 학습 모드 ===
# True: val holdout 없이 전체 데이터로 학습 (최종 제출용)
# False: 90/10 split으로 학습 (실험/검증용)
FULL_DATA_MODE = True
FULL_DATA_EPOCHS = 3  # 전체 데이터 모드에서 고정 epoch (검증 없으므로 보수적)

EARLY_STOPPING_PATIENCE = 1  # stop if val loss increases for 1 consecutive epoch
TIMING_CHECKPOINT_STEP = 100  # print time estimate after this many steps
GPU_MEMORY_WARN_THRESHOLD_GB = 70  # warn if peak memory > this after step 1
VAL_ACCURACY_FALLBACK_THRESHOLD = 0.87  # do NOT use model if val acc < this

# -- Optional Model B Hyperparameters (Phase 5) --
SEED_B = 123
LR_B = 5e-5
LORA_R_B = 64
LORA_ALPHA_B = 128

# -- TTA Permutations --
TTA_PERMUTATIONS = [
    ["a", "b", "c", "d"],  # original
    ["b", "d", "a", "c"],  # shuffled 1
    ["c", "a", "d", "b"],  # shuffled 2
    ["d", "c", "b", "a"],  # shuffled 3
]

# -- Paths (auto-detect Colab vs local) --
SAVE_DIR_A = "./checkpoints/model_a"
SAVE_DIR_B = "./checkpoints/model_b"



## Cell 3: Data Loading and Dev Filtering

Load train.csv and dev.csv. Apply majority vote to dev.csv, keep only rows with >=3/5 annotator agreement.

# 데이터 다운로드 (Kaggle CLI 또는 Google Drive)

**방법 1 (권장)**: Kaggle CLI로 직접 다운로드 (Drive 용량 불필요, ~1-2분)  
**방법 2**: Google Drive에서 복사 (기존 방식)

In [ ]:
# ============================================================
# 데이터는 Google Drive에서 로드합니다.
# 이 셀은 건너뛰세요. 아래 Drive 마운트 셀에서 처리됩니다.
# ============================================================
print('데이터는 Drive 마운트 셀에서 로드됩니다.')


# Google Drive 마운트 & 데이터 준비

Colab 환경에서는 Google Drive에서 데이터를 로드합니다.  
로컬 서버 환경에서는 이 셀을 건너뛰세요.

In [ ]:
# ============================================================
# Google Drive 마운트 & 데이터 로드
# ============================================================

import os

from google.colab import drive
drive.mount('/content/drive')

# ★ Google Drive 데이터 경로 ★
# /content/drive/MyDrive/ 바로 아래에 train/, dev/, test/ 폴더와 CSV 파일이 있는 구조
DRIVE_ROOT = '/content/drive/MyDrive'

# Drive → 로컬 복사 (I/O 속도 향상)
DATA_DIR = '/content/data'
if not os.path.exists(os.path.join(DATA_DIR, 'train.csv')):
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Drive → 로컬 복사 중...')
    !cp "{DRIVE_ROOT}/train.csv" "{DATA_DIR}/"
    !cp "{DRIVE_ROOT}/dev.csv" "{DATA_DIR}/"
    !cp "{DRIVE_ROOT}/test.csv" "{DATA_DIR}/"
    !cp -r "{DRIVE_ROOT}/train" "{DATA_DIR}/"
    !cp -r "{DRIVE_ROOT}/dev" "{DATA_DIR}/"
    !cp -r "{DRIVE_ROOT}/test" "{DATA_DIR}/"
    print('복사 완료!')
else:
    print(f'로컬 데이터 이미 존재: {DATA_DIR}')

# 체크포인트 백업 경로
DRIVE_CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

# CSV 경로 (CSV가 DATA_DIR 바로 아래)
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
DEV_CSV = os.path.join(DATA_DIR, 'dev.csv')
TEST_CSV = os.path.join(DATA_DIR, 'test.csv')

# 확인
for name, path in [('Train', TRAIN_CSV), ('Dev', DEV_CSV), ('Test', TEST_CSV)]:
    print(f'  {name}: {path} [{"OK" if os.path.exists(path) else "NOT FOUND"}]')
for split in ['train', 'dev', 'test']:
    img_dir = os.path.join(DATA_DIR, split)
    if os.path.exists(img_dir):
        print(f'  {split}/ images: {len(os.listdir(img_dir))} files')


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
dev_df = pd.read_csv(DEV_CSV)

# Fix image paths to absolute
train_df["path"] = train_df["path"].apply(lambda p: os.path.join(DATA_DIR, p))
dev_df["path"] = dev_df["path"].apply(lambda p: os.path.join(DATA_DIR, p))

print(f"Raw train: {train_df.shape}")
print(f"Raw dev: {dev_df.shape}")

# Majority vote for dev data
def majority_vote(row):
    votes = [row[f"answer{i}"] for i in range(1, 6)]
    counter = Counter(votes)
    most_common, count = counter.most_common(1)[0]
    return most_common if count >= DEV_AGREEMENT_THRESHOLD else None

dev_df["answer"] = dev_df.apply(majority_vote, axis=1)

# Agreement distribution before filtering
agreement_counts = []
for _, row in pd.read_csv(DEV_CSV).iterrows():
    votes = [row[f"answer{i}"] for i in range(1, 6)]
    max_count = Counter(votes).most_common(1)[0][1]
    agreement_counts.append(max_count)
print(f"\nAgreement distribution: {Counter(agreement_counts)}")

dev_df = dev_df.dropna(subset=["answer"])

# Keep only columns matching train_df
keep_cols = ["id", "path", "question", "a", "b", "c", "d", "answer"]
dev_df = dev_df[keep_cols]

print(f"\nFiltered dev (>={DEV_AGREEMENT_THRESHOLD}/5 agreement): {dev_df.shape}")
print(f"Train columns: {list(train_df.columns)}")
print(f"Dev columns: {list(dev_df.columns)}")
assert dev_df["answer"].isna().sum() == 0, "NaN answers remain!"

## Cell 4: Question Type Classification

Classify each question into types (counting, material, object_id, other) for stratified sampling.

In [ ]:
def classify_question(question: str) -> str:
    q = question.lower()
    counting_keywords = ["몇 개", "몇개", "개수", "몇 병", "몇 캔", "총 몇"]
    material_keywords = ["재질", "재료", "소재", "무엇으로 만들", "어떤 재질"]

    if any(kw in q for kw in counting_keywords):
        return "counting"
    elif any(kw in q for kw in material_keywords):
        return "material"
    else:
        if "무엇" in q or "어떤 것" in q or "종류" in q:
            return "object_id"
        return "other"

train_df["q_type"] = train_df["question"].apply(classify_question)
dev_df["q_type"] = dev_df["question"].apply(classify_question)

print("Train question type distribution:")
print(train_df["q_type"].value_counts())
print(f"\nTrain distribution (normalized):")
print(train_df["q_type"].value_counts(normalize=True))
print(f"\nDev question type distribution:")
print(dev_df["q_type"].value_counts())
print(f"\nDev distribution (normalized):")
print(dev_df["q_type"].value_counts(normalize=True))

## Cell 5: Dev Data Stratified Sampling and Merge

Subsample dev data so its question-type distribution matches train, then merge.
Expected usable dev rows: ~1,347 (bottleneck on underrepresented types).

In [ ]:
train_dist = train_df["q_type"].value_counts(normalize=True)

# Find the bottleneck q_type (smallest dev_count / train_ratio)
dev_type_counts = dev_df["q_type"].value_counts()
max_total = min(
    dev_type_counts[qt] / train_dist[qt]
    for qt in train_dist.index
    if qt in dev_type_counts
)
max_total = int(max_total)

print(f"Bottleneck analysis:")
for qt in train_dist.index:
    if qt in dev_type_counts:
        limit = dev_type_counts[qt] / train_dist[qt]
        print(f"  {qt}: dev_count={dev_type_counts[qt]}, train_ratio={train_dist[qt]:.3f}, max_total={limit:.0f}")
print(f"\nBottleneck max_total: {max_total}")

sampled_dev = []
for qt in train_dist.index:
    n = int(max_total * train_dist[qt])
    qt_df = dev_df[dev_df["q_type"] == qt]
    if len(qt_df) >= n:
        sampled_dev.append(qt_df.sample(n=n, random_state=SEED_A))
    else:
        sampled_dev.append(qt_df)
    print(f"  Sampled {qt}: {min(n, len(qt_df))} rows (target={n}, available={len(qt_df)})")

dev_sampled = pd.concat(sampled_dev, ignore_index=True)

# Merge (drop q_type before merge, re-add later)
train_for_merge = train_df.drop(columns=["q_type"])
dev_for_merge = dev_sampled.drop(columns=["q_type"])
combined_df = pd.concat([train_for_merge, dev_for_merge], ignore_index=True)

print(f"\nFinal combined_df: {combined_df.shape}")
print(f"  Train rows: {len(train_df)}")
print(f"  Dev sampled rows: {len(dev_sampled)}")
print(f"  Total: {len(combined_df)}")

## Cell 6: Train/Val Split

90/10 stratified split using question type as stratification key.

In [ ]:
# ============================================================
# Train/Val 분리 (또는 전체 데이터 모드)
# ============================================================

combined_df['q_type'] = combined_df['question'].apply(classify_question)

if FULL_DATA_MODE:
    # 전체 데이터로 학습 (최종 제출용)
    final_train_df = combined_df.reset_index(drop=True)
    # 모니터링용 소규모 val set (학습 데이터와 겹침 — early stopping에 사용하지 않음)
    final_val_df = combined_df.sample(n=min(100, len(combined_df)), random_state=SEED_A).reset_index(drop=True)
    NUM_EPOCHS = FULL_DATA_EPOCHS  # 고정 epoch
    EARLY_STOPPING_PATIENCE = 999  # 사실상 비활성화
    print(f'FULL DATA MODE: Train={len(final_train_df)} (전체), Val={len(final_val_df)} (모니터링용)')
    print(f'Fixed epochs: {NUM_EPOCHS} (no early stopping)')
else:
    # 90/10 Stratified split (실험/검증용)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED_A)
    train_idx, val_idx = next(sss.split(combined_df, combined_df['q_type']))
    final_train_df = combined_df.iloc[train_idx].reset_index(drop=True)
    final_val_df = combined_df.iloc[val_idx].reset_index(drop=True)
    print(f'SPLIT MODE: Train={len(final_train_df)}, Val={len(final_val_df)}')

print(f'Train q_type dist:\n{final_train_df["q_type"].value_counts(normalize=True)}')


## Cell 7: Domain-Specific System Prompt

Korean-language system prompt specialized for recycling VQA.
Includes counting strategy, material categories, and metal-vs-plastic disambiguation.
Also defines `build_mc_prompt` and `build_mc_prompt_shuffled` for TTA.

In [ ]:
SYSTEM_PROMPT = (
    "당신은 재활용 분류 전문가입니다. "
    "재활용품 이미지를 보고 질문에 정확히 답변합니다.\n\n"
    "핵심 지침:\n"
    "1. 개수를 세는 질문: 이미지를 꼼꼼히 살펴보고 해당 물품의 정확한 개수를 세세요. "
    "겹치거나 부분적으로 보이는 물품도 포함합니다.\n"
    "2. 재질 판별: 플라스틱(투명/불투명, 유연/경질), 유리(투명/색유리), "
    "금속(알루미늄/철), 종이(골판지/일반), 비닐/스티로폼을 구분하세요.\n"
    "3. 금속 vs 플라스틱: 광택, 반사, 찌그러짐 패턴으로 구분합니다. "
    "금속은 광택이 있고 찌그러지면 주름이 생깁니다. "
    "플라스틱은 무광이며 라벨이 붙어있는 경우가 많습니다.\n\n"
    "반드시 a, b, c, d 중 하나의 소문자 한 글자로만 답하세요."
)


def build_mc_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )


def build_mc_prompt_shuffled(
    question: str, options: dict, permutation: list
) -> Tuple[str, dict]:
    """Build MC prompt with shuffled answer positions.

    Args:
        question: The question text
        options: Dict mapping original keys to option text, e.g. {"a": "text1", ...}
        permutation: List of original keys in shuffled order, e.g. ["b", "d", "a", "c"]

    Returns:
        (prompt_text, position_to_original_map)
        position_to_original_map: {"a": "b", "b": "d", ...} meaning position "a" holds original option "b"
    """
    position_labels = ["a", "b", "c", "d"]
    position_to_original = {}
    lines = [question]
    for pos_label, orig_key in zip(position_labels, permutation):
        lines.append(f"({pos_label}) {options[orig_key]}")
        position_to_original[pos_label] = orig_key
    lines.append("")
    lines.append("정답을 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")
    return "\n".join(lines), position_to_original


print("System prompt defined.")
print(f"Prompt length: {len(SYSTEM_PROMPT)} chars")
# Quick test
test_prompt = build_mc_prompt("이 이미지에서 플라스틱 병은 몇 개인가요?", "1개", "2개", "3개", "4개")
print(f"\nSample prompt:\n{test_prompt}")

## Cell 8: VQAMCDataset Class

Custom Dataset that loads images and builds chat messages for the collator.

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor, train: bool = True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")

        # 학습 시에만 보수적 이미지 증강 적용
        if self.train:
            import numpy as np
            img_np = np.array(img)
            img_np = TRAIN_TRANSFORM(image=img_np)["image"]
            img = Image.fromarray(img_np)

        options = {
            "a": str(row["a"]),
            "b": str(row["b"]),
            "c": str(row["c"]),
            "d": str(row["d"]),
        }

        if self.train:
            # 학습 시 답변 순서를 랜덤으로 치환하여 위치 편향 방지
            gold_orig = str(row["answer"]).strip().lower()
            keys = list(options.keys())
            random.shuffle(keys)  # ["c","a","d","b"] 등
            # 새 위치에서의 프롬프트 생성
            shuffled_prompt, pos_to_orig = build_mc_prompt_shuffled(
                str(row["question"]), options, keys
            )
            # gold label을 새 위치로 매핑: 원래 "b"가 정답이었으면 새 위치 찾기
            orig_to_pos = {v: k for k, v in pos_to_orig.items()}
            gold_new = orig_to_pos[gold_orig]
            user_text = shuffled_prompt
        else:
            # 추론 시에는 원래 순서 유지
            user_text = build_mc_prompt(
                str(row["question"]),
                str(row["a"]), str(row["b"]),
                str(row["c"]), str(row["d"]),
            )

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text},
            ]},
        ]
        if self.train:
            messages.append(
                {"role": "assistant", "content": [{"type": "text", "text": gold_new}]}
            )

        return {"messages": messages, "image": img}

print("VQAMCDataset defined (with training-time answer shuffling).")

## Cell 9: DataCollator with Label Masking

Collator applies chat template, tokenizes, and masks labels so loss is only computed on the assistant answer token.

In [ ]:
@dataclass
class VQACollator:
    processor: Any
    train: bool = True

    def __call__(self, batch: List[Dict]) -> Dict[str, torch.Tensor]:
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=(not self.train),
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text=texts, images=images,
            padding=True, return_tensors="pt",
        )

        if self.train:
            labels = enc["input_ids"].clone()
            # Mask pad tokens
            pad_id = self.processor.tokenizer.pad_token_id
            if pad_id is not None:
                labels[labels == pad_id] = -100

            # For each sequence, mask all prompt tokens (keep only assistant answer)
            for i in range(labels.shape[0]):
                prompt_msgs = batch[i]["messages"][:-1]  # without assistant
                prompt_text = self.processor.apply_chat_template(
                    prompt_msgs, tokenize=False, add_generation_prompt=True,
                )
                prompt_enc = self.processor(
                    text=[prompt_text], images=[batch[i]["image"]],
                    return_tensors="pt",
                )
                prompt_len = prompt_enc["input_ids"].shape[1]
                labels[i, :prompt_len] = -100

            enc["labels"] = labels

        return enc

print("VQACollator defined.")

## Cell 10: Model Loading Functions

Reusable functions to load model with LoRA config (for training) and load saved adapter (for inference).
Uses bf16 full precision with flash_attention_2 and gradient checkpointing.

In [ ]:
def ensure_checkpoint(save_dir: str):
    """로컬에 체크포인트가 없으면 Drive 백업에서 복구"""
    if os.path.exists(save_dir) and os.listdir(save_dir):
        print(f'Checkpoint found locally: {save_dir}')
        return True

    if DRIVE_CHECKPOINT_DIR is None:
        print(f'WARNING: No checkpoint at {save_dir} and no Drive backup configured')
        return False

    # Drive에서 복구 시도
    model_name = os.path.basename(save_dir)  # 'model_a' or 'model_b'
    drive_path = os.path.join(DRIVE_CHECKPOINT_DIR, model_name)

    if os.path.exists(drive_path):
        print(f'Restoring checkpoint from Drive: {drive_path} -> {save_dir}')
        import shutil
        os.makedirs(save_dir, exist_ok=True)
        shutil.copytree(drive_path, save_dir, dirs_exist_ok=True)
        print(f'Restore complete: {os.listdir(save_dir)}')
        return True
    else:
        print(f'WARNING: No backup found at {drive_path}')
        return False


def load_model_and_processor(
    model_id: str, image_size: int,
    lora_r: int, lora_alpha: int,
    lora_dropout: float, seed: int,
):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    processor = AutoProcessor.from_pretrained(
        model_id,
        min_pixels=image_size * image_size,
        max_pixels=image_size * image_size,
        trust_remote_code=True,
    )

    base_model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto",
        trust_remote_code=True,
    )
    base_model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=lora_r, lora_alpha=lora_alpha,
        lora_dropout=lora_dropout, bias="none",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    return model, processor


def load_model_for_inference(save_dir: str):
    """Load base model + saved LoRA adapter for inference."""
    if not ensure_checkpoint(save_dir):
        raise FileNotFoundError(f'No checkpoint found at {save_dir} (local or Drive)')
    processor = AutoProcessor.from_pretrained(
        save_dir,
        min_pixels=IMAGE_SIZE * IMAGE_SIZE,
        max_pixels=IMAGE_SIZE * IMAGE_SIZE,
        trust_remote_code=True,
    )
    base_model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, save_dir)
    model.eval()
    return model, processor

print("Model loading functions defined.")

## Cell 11: Training Function

Complete training loop with:
- Cosine schedule with warmup
- Gradient clipping
- Per-epoch validation and best-model saving
- Early stopping (patience=1)
- GPU memory probe after step 1
- Timing checkpoint after step 100

In [ ]:
def train_model(
    model, processor,
    train_df: pd.DataFrame, val_df: pd.DataFrame,
    num_epochs: int, lr: float, grad_accum: int,
    save_dir: str, seed: int,
):
    random.seed(seed)
    torch.manual_seed(seed)

    train_ds = VQAMCDataset(train_df, processor, train=True)
    val_ds = VQAMCDataset(val_df, processor, train=True)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=VQACollator(processor, train=True),
        num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=VQACollator(processor, train=True),
        num_workers=2, pin_memory=True,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = num_epochs * math.ceil(len(train_loader) / grad_accum)
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, warmup_steps, total_steps
    )

    best_val_loss = float("inf")
    patience_counter = 0
    os.makedirs(save_dir, exist_ok=True)
    train_start_time = time.time()

    for epoch in range(num_epochs):
        # -- Train --
        model.train()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [train]")

        for step, batch in enumerate(pbar, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum

            loss.backward()
            running_loss += loss.item()

            # === GPU Memory Probe after step 1 ===
            if epoch == 0 and step == 1:
                peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
                print(f"\n[MEMORY PROBE] Peak GPU memory after step 1: {peak_mem_gb:.1f}GB")
                if peak_mem_gb > GPU_MEMORY_WARN_THRESHOLD_GB:
                    print(f"  WARNING: Peak memory ({peak_mem_gb:.1f}GB) exceeds {GPU_MEMORY_WARN_THRESHOLD_GB}GB threshold!")
                    print(f"  Consider fallback: reduce IMAGE_SIZE to 560px or reduce LORA_R")
                else:
                    print(f"  OK: Memory usage within safe range ({peak_mem_gb:.1f}GB / 80GB)")

            if step % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), MAX_GRAD_NORM
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                pbar.set_postfix({
                    "loss": f"{running_loss:.4f}",
                    "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                })
                running_loss = 0.0

            # === Timing Checkpoint after step 100 ===
            if epoch == 0 and step == TIMING_CHECKPOINT_STEP:
                elapsed = time.time() - train_start_time
                per_step = elapsed / step
                total_train_steps = num_epochs * len(train_loader)
                estimated_total = per_step * total_train_steps
                print(f"\n[TIMING CHECKPOINT] After {step} steps:")
                print(f"  Per-step time: {per_step:.2f}s")
                print(f"  Estimated total training time: {estimated_total/3600:.1f}h")
                print(f"  Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
                if per_step > 2.5:
                    print(f"  WARNING: Per-step time ({per_step:.2f}s) exceeds 2.5s threshold!")
                    print(f"  Consider: reduce NUM_EPOCHS to 2, or reduce IMAGE_SIZE to 560px")

        # Handle remaining gradient accumulation steps
        if step % grad_accum != 0:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), MAX_GRAD_NORM
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        # -- Validate --
        model.eval()
        val_loss, val_steps = 0.0, 0
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                for vb in tqdm(val_loader, desc=f"Epoch {epoch+1} [valid]"):
                    vb = {k: v.to(device) for k, v in vb.items()}
                    val_loss += model(**vb).loss.item()
                    val_steps += 1

        avg_val_loss = val_loss / max(val_steps, 1)
        print(f"[Epoch {epoch+1}] val_loss={avg_val_loss:.4f} | best={best_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            model.save_pretrained(save_dir)
            processor.save_pretrained(save_dir)
            print(f"  -> Saved best model to {save_dir}")
        else:
            patience_counter += 1
            print(f"  -> Val loss did not improve. Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  -> EARLY STOPPING triggered at epoch {epoch+1}")
                break

    total_train_time = time.time() - train_start_time
    print(f"\nTraining completed in {total_train_time/3600:.1f}h")
    return best_val_loss

print("Training function defined.")

## Cell 12: Token ID Verification

Pre-verify that "a", "b", "c", "d" each map to a single token ID. Critical for logprob inference.

In [ ]:
temp_processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = temp_processor.tokenizer

for letter in ["a", "b", "c", "d"]:
    ids = tokenizer.encode(letter, add_special_tokens=False)
    print(f"'{letter}' -> token_ids={ids}, decoded='{tokenizer.decode(ids)}'")
    assert len(ids) == 1, f"Expected single token for '{letter}', got {ids}"

ANSWER_TOKEN_IDS = {
    letter: tokenizer.encode(letter, add_special_tokens=False)[0]
    for letter in ["a", "b", "c", "d"]
}
print(f"\nAnswer token ID mapping: {ANSWER_TOKEN_IDS}")
del temp_processor, tokenizer
gc.collect()

## Cell 13: Train Model A

Load and train the first model (seed=42, lr=2e-5, LoRA r=32, alpha=64).

In [ ]:
# ============================================================
# Model A 학습 (체크포인트 있으면 건너뛰기)
# ============================================================

# FULL_DATA_MODE에서 재학습이 필요하면 아래를 False로 변경
# 이전 체크포인트와 다른 설정(전체 데이터)으로 학습하므로 기본 False
SKIP_TRAINING_A = False

# 로컬 또는 Drive에 체크포인트가 있는지 확인
if os.path.exists(SAVE_DIR_A) and any(f.endswith('.safetensors') for f in os.listdir(SAVE_DIR_A)):
    print(f'Model A checkpoint already exists at {SAVE_DIR_A}')
    SKIP_TRAINING_A = True
elif DRIVE_CHECKPOINT_DIR:
    drive_model_a = os.path.join(DRIVE_CHECKPOINT_DIR, 'model_a')
    if os.path.exists(drive_model_a) and os.listdir(drive_model_a):
        print(f'Model A checkpoint found in Drive: {drive_model_a}')
        import shutil
        os.makedirs(SAVE_DIR_A, exist_ok=True)
        shutil.copytree(drive_model_a, SAVE_DIR_A, dirs_exist_ok=True)
        print(f'Restored to local: {SAVE_DIR_A}')
        SKIP_TRAINING_A = True


# ★ 전체 데이터 재학습: 체크포인트가 있어도 강제 재학습
if FULL_DATA_MODE:
    SKIP_TRAINING_A = False
    print("FULL_DATA_MODE: 전체 데이터로 재학습합니다.")

if SKIP_TRAINING_A:
    print('=' * 60)
    print('SKIPPING TRAINING — checkpoint already exists!')
    print('학습을 다시 하려면 SKIP_TRAINING_A = False 로 변경하세요.')
    print('=' * 60)
else:
    print('=' * 60)
    print('TRAINING MODEL A: seed=42, lr=2e-5, LoRA r=32, alpha=64')
    print(f'Dataset: {len(final_train_df)} train, {len(final_val_df)} val')
    print(f'Epochs: {NUM_EPOCHS} (early stopping patience={EARLY_STOPPING_PATIENCE})')
    print('=' * 60)

    model_a, processor_a = load_model_and_processor(
        MODEL_ID, IMAGE_SIZE,
        lora_r=LORA_R_A, lora_alpha=LORA_ALPHA_A,
        lora_dropout=LORA_DROPOUT, seed=SEED_A,
    )

    best_loss_a = train_model(
        model_a, processor_a,
        final_train_df, final_val_df,
        num_epochs=NUM_EPOCHS, lr=LR_A,
        grad_accum=GRAD_ACCUM, save_dir=SAVE_DIR_A,
        seed=SEED_A,
    )
    print(f'Model A best val loss: {best_loss_a:.4f}')

    # Drive에 체크포인트 백업
    if DRIVE_CHECKPOINT_DIR:
        import shutil
        backup_path = os.path.join(DRIVE_CHECKPOINT_DIR, 'model_a')
        if os.path.exists(backup_path):
            shutil.rmtree(backup_path)
        shutil.copytree(SAVE_DIR_A, backup_path)
        print(f'Model A checkpoint backed up to Drive: {backup_path}')

    # 학습 후 GPU 메모리 정리
    del model_a, processor_a
    gc.collect()
    torch.cuda.empty_cache()
    print('GPU memory cleaned after training.')


## Cell 14: Logprob Inference Function

Forward-pass logprob extraction instead of `model.generate()`. Returns predictions and optionally raw logprobs (needed for TTA averaging and ensemble).

In [ ]:
def logprob_inference(
    model, processor,
    df: pd.DataFrame,
    answer_token_ids: dict,
    return_logprobs: bool = False,
):
    """Forward-pass logprob extraction instead of generate().

    If return_logprobs=True, returns (predictions, logprobs_list)
    where logprobs_list[i] = {"a": float, "b": float, "c": float, "d": float}
    """
    model.eval()
    predictions = []
    all_logprobs = []
    ds = VQAMCDataset(df, processor, train=False)

    for idx in tqdm(range(len(ds)), desc="Logprob Inference"):
        sample = ds[idx]
        text = processor.apply_chat_template(
            sample["messages"], tokenize=False,
            add_generation_prompt=True,
        )
        inputs = processor(
            text=[text], images=[sample["image"]],
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**inputs)

        # Logits at last position
        last_logits = outputs.logits[0, -1, :]

        # Extract logprobs for a, b, c, d
        logprobs = {
            letter: last_logits[tid].item()
            for letter, tid in answer_token_ids.items()
        }

        pred = max(logprobs, key=logprobs.get)
        predictions.append(pred)
        if return_logprobs:
            all_logprobs.append(logprobs)

    if return_logprobs:
        return predictions, all_logprobs
    return predictions

print("Logprob inference function defined.")

# vLLM 배치 추론 (HF 대비 2-4배 빠름)

학습은 HuggingFace로 완료했으므로, 추론만 vLLM으로 전환합니다.
- LoRA 어댑터를 vLLM에 로딩
- 전체 테스트셋을 한 번에 배치 처리
- TTA도 모든 permutation을 한 번에 처리 (5,074×4 = 20,296건)
- 실패 시 위의 HF logprob_inference로 자동 fallback

In [ ]:
# ============================================================
# vLLM 배치 추론 함수 (LoRA 어댑터 지원)
# ============================================================

USE_VLLM = True  # vLLM 사용 여부 (실패 시 자동으로 False로 전환)

try:
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    print('vLLM imported successfully')
except ImportError:
    print('vLLM not available, will use HF fallback')
    USE_VLLM = False


def init_vllm(model_id: str, lora_dir: str, image_size: int, max_lora_rank: int = 64):
    """vLLM 엔진 초기화 + LoRA 어댑터 로딩"""
    llm = LLM(
        model=model_id,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        trust_remote_code=True,
        dtype="bfloat16",
        max_model_len=4096,
        gpu_memory_utilization=0.85,
        enforce_eager=True,
        tensor_parallel_size=1,
        limit_mm_per_prompt={"image": 1},
        mm_processor_kwargs={
            "min_pixels": image_size * image_size,
            "max_pixels": image_size * image_size,
        },
    )
    lora_req = LoRARequest("vqa_adapter", 1, lora_dir)
    sampling = SamplingParams(max_tokens=1, temperature=0, logprobs=20)
    return llm, lora_req, sampling


def vllm_logprob_inference(
    llm, lora_req, sampling, processor,
    df, answer_token_ids: dict,
) -> Tuple[List[str], List[dict]]:
    """vLLM 배치 추론: 전체 데이터셋을 한 번에 처리"""
    # 1) 모든 입력을 리스트로 준비
    vllm_inputs = []
    for row in tqdm(df.itertuples(), total=len(df), desc='Preparing vLLM inputs'):
        img = Image.open(row.path).convert('RGB')
        user_text = build_mc_prompt(str(row.question), str(row.a), str(row.b), str(row.c), str(row.d))
        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': user_text},
            ]},
        ]
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        vllm_inputs.append({'prompt': prompt, 'multi_modal_data': {'image': img}})

    # 2) 배치 추론 (이 한 줄이 전부!)
    print(f'Running vLLM batch inference on {len(vllm_inputs)} samples...')
    outputs = llm.generate(vllm_inputs, sampling, lora_request=lora_req)

    # 3) logprob 추출
    tokenizer = processor.tokenizer
    predictions = []
    all_logprobs = []
    for output in outputs:
        token_logprobs = output.outputs[0].logprobs[0]  # 첫 번째 (유일한) 생성 토큰의 logprobs
        lp = {}
        for letter, tid in answer_token_ids.items():
            if tid in token_logprobs:
                lp[letter] = token_logprobs[tid].logprob
            else:
                lp[letter] = -100.0  # top-20에 없으면 매우 낮은 확률
        pred = max(lp, key=lp.get)
        predictions.append(pred)
        all_logprobs.append(lp)

    return predictions, all_logprobs


def vllm_tta_inference(
    llm, lora_req, sampling, processor,
    df, answer_token_ids: dict, permutations: list,
) -> Tuple[List[str], List[dict]]:
    """vLLM TTA: 모든 permutation을 한 번에 배치 처리"""
    n_samples = len(df)
    n_perms = len(permutations)

    # 1) 모든 (샘플 × permutation) 입력 준비
    vllm_inputs = []
    perm_mappings = []  # (sample_idx, pos_to_orig) 매핑 보관

    for idx, row in tqdm(enumerate(df.itertuples()), total=n_samples, desc='Preparing TTA inputs'):
        img = Image.open(row.path).convert('RGB')
        options = {'a': str(row.a), 'b': str(row.b), 'c': str(row.c), 'd': str(row.d)}

        for perm in permutations:
            prompt_text, pos_to_orig = build_mc_prompt_shuffled(str(row.question), options, perm)
            messages = [
                {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
                {'role': 'user', 'content': [
                    {'type': 'image', 'image': img},
                    {'type': 'text', 'text': prompt_text},
                ]},
            ]
            prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            vllm_inputs.append({'prompt': prompt, 'multi_modal_data': {'image': img}})
            perm_mappings.append((idx, pos_to_orig))

    # 2) 전체 배치 추론
    print(f'Running vLLM TTA batch: {len(vllm_inputs)} inputs ({n_samples} samples × {n_perms} perms)...')
    outputs = llm.generate(vllm_inputs, sampling, lora_request=lora_req)

    # 3) 결과를 샘플별로 재조립 + logprob 평균
    sample_logprobs = [{'a': 0.0, 'b': 0.0, 'c': 0.0, 'd': 0.0} for _ in range(n_samples)]

    for out_idx, output in enumerate(outputs):
        sample_idx, pos_to_orig = perm_mappings[out_idx]
        token_logprobs = output.outputs[0].logprobs[0]

        for pos_label in ['a', 'b', 'c', 'd']:
            tid = answer_token_ids[pos_label]
            lp = token_logprobs[tid].logprob if tid in token_logprobs else -100.0
            orig_key = pos_to_orig[pos_label]
            sample_logprobs[sample_idx][orig_key] += lp

    # 평균 + 예측
    predictions = []
    averaged = []
    for lps in sample_logprobs:
        avg = {k: v / n_perms for k, v in lps.items()}
        predictions.append(max(avg, key=avg.get))
        averaged.append(avg)

    return predictions, averaged


print('vLLM inference functions defined.')


## Cell 15: Validation Accuracy + Error Analysis + Fallback Guard

Compute accuracy on val set using logprob inference. Break down by question type.
Fallback guard triggers if accuracy < 0.87.

In [ ]:
# Reload best Model A
del model_a; gc.collect(); torch.cuda.empty_cache()
model_a_best, proc_a = load_model_for_inference(SAVE_DIR_A)

val_preds_a = logprob_inference(model_a_best, proc_a, final_val_df, ANSWER_TOKEN_IDS)

# Overall accuracy
val_gold = final_val_df["answer"].str.strip().str.lower().tolist()
correct_a = sum(p == g for p, g in zip(val_preds_a, val_gold))
val_accuracy_a = correct_a / len(val_gold)
print(f"\nModel A Val Accuracy: {correct_a}/{len(val_gold)} = {val_accuracy_a:.4f}")

# === FALLBACK GUARD ===
if val_accuracy_a < VAL_ACCURACY_FALLBACK_THRESHOLD:
    print(f"\n{'='*60}")
    print(f"FALLBACK GUARD TRIGGERED: Val accuracy ({val_accuracy_a:.4f}) < {VAL_ACCURACY_FALLBACK_THRESHOLD}")
    print(f"WARNING: Do NOT use this model's predictions for final submission!")
    print(f"Consider: re-check data pipeline, reduce LR, increase epochs")
    print(f"{'='*60}")
else:
    print(f"\nFallback guard OK: Val accuracy ({val_accuracy_a:.4f}) >= {VAL_ACCURACY_FALLBACK_THRESHOLD}")

# Per question-type breakdown
final_val_df["pred_a"] = val_preds_a
final_val_df["correct_a"] = (
    final_val_df["pred_a"] == final_val_df["answer"].str.strip().str.lower()
)
print("\nPer-type accuracy:")
print(final_val_df.groupby("q_type")["correct_a"].mean())

# Error confusion: predicted vs actual answer position
confusion = Counter()
for _, row in final_val_df.iterrows():
    actual = row["answer"].strip().lower()
    pred = row["pred_a"]
    confusion[(actual, pred)] += 1

print("\nErrors (actual -> predicted): count")
for (actual, pred), cnt in sorted(confusion.items()):
    if actual != pred:
        print(f"  {actual} -> {pred}: {cnt}")

## Cell 16: TTA Inference Function

TTA with 4 answer-choice permutations. For each question, runs 4 forward passes with shuffled answer positions, maps logprobs back to original answer options, and averages.

In [ ]:
def tta_logprob_inference(
    model, processor,
    df: pd.DataFrame,
    answer_token_ids: dict,
    permutations: list,
) -> Tuple[List[str], List[dict]]:
    """TTA with answer-choice shuffling.

    For each question, runs len(permutations) forward passes with different
    answer orderings. Logprobs are mapped back to original answer options
    and averaged across permutations.

    Returns:
        (predictions, averaged_logprobs)
    """
    model.eval()
    predictions = []
    averaged_logprobs_list = []

    for idx in tqdm(range(len(df)), desc="TTA Inference"):
        row = df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        options = {
            "a": str(row["a"]),
            "b": str(row["b"]),
            "c": str(row["c"]),
            "d": str(row["d"]),
        }

        # Accumulate logprobs per original answer option
        orig_logprobs = {"a": 0.0, "b": 0.0, "c": 0.0, "d": 0.0}

        for perm in permutations:
            prompt_text, pos_to_orig = build_mc_prompt_shuffled(
                str(row["question"]), options, perm
            )

            messages = [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
                {"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt_text},
                ]},
            ]

            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            inputs = processor(
                text=[text], images=[img],
                return_tensors="pt",
            ).to(device)

            with torch.no_grad():
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    outputs = model(**inputs)

            last_logits = outputs.logits[0, -1, :]

            # Get logprobs for position labels a,b,c,d
            for pos_label in ["a", "b", "c", "d"]:
                logprob = last_logits[answer_token_ids[pos_label]].item()
                # Map position back to original answer option
                orig_key = pos_to_orig[pos_label]
                orig_logprobs[orig_key] += logprob

        # Average across permutations
        n_perm = len(permutations)
        avg_logprobs = {k: v / n_perm for k, v in orig_logprobs.items()}

        pred = max(avg_logprobs, key=avg_logprobs.get)
        predictions.append(pred)
        averaged_logprobs_list.append(avg_logprobs)

    return predictions, averaged_logprobs_list

print("TTA inference function defined.")

## Cell 17: Test Inference - Single Model (Model A)

Run logprob inference on test set for Model A (no TTA). Baseline submission.

In [ ]:
# ============================================================
# Test Inference — Single Model (vLLM 배치 or HF fallback)
# ============================================================

test_df = pd.read_csv(TEST_CSV)
test_df['path'] = test_df['path'].apply(lambda p: os.path.join(DATA_DIR, p))

if USE_VLLM:
    try:
        # vLLM 엔진 초기화 (LoRA 어댑터 로딩)
        print('Initializing vLLM with LoRA adapter...')
        vllm_engine, lora_req, sampling = init_vllm(
            MODEL_ID, SAVE_DIR_A, IMAGE_SIZE, max_lora_rank=max(LORA_R_A, LORA_R_B)
        )
        vllm_processor = AutoProcessor.from_pretrained(
            MODEL_ID,
            min_pixels=IMAGE_SIZE * IMAGE_SIZE,
            max_pixels=IMAGE_SIZE * IMAGE_SIZE,
            trust_remote_code=True,
        )
        test_preds_a, test_logprobs_a = vllm_logprob_inference(
            vllm_engine, lora_req, sampling, vllm_processor,
            test_df, ANSWER_TOKEN_IDS,
        )
        print(f'vLLM inference complete: {Counter(test_preds_a)}')
    except Exception as e:
        print(f'vLLM failed: {e}')
        print('Falling back to HF inference...')
        USE_VLLM = False
        test_preds_a = logprob_inference(model_a_best, proc_a, test_df, ANSWER_TOKEN_IDS)
else:
    # Model A is already loaded from validation cell
    test_preds_a = logprob_inference(model_a_best, proc_a, test_df, ANSWER_TOKEN_IDS)

print(f'Model A predictions: {Counter(test_preds_a)}')
sub_a = pd.DataFrame({'id': test_df['id'], 'answer': test_preds_a})
sub_a.to_csv('submission_model_a.csv', index=False)
print('Saved submission_model_a.csv')


## Cell 18: TTA Inference on Test Set

Run TTA with 4 answer-choice permutations on the test set. This is the primary submission.

In [ ]:
# ============================================================
# TTA Inference (vLLM 배치 or HF fallback)
# ============================================================

print('=' * 60)
print(f'TTA INFERENCE: {len(test_df)} rows x {len(TTA_PERMUTATIONS)} permutations')
print('=' * 60)

tta_start = time.time()

if USE_VLLM:
    try:
        # vLLM은 이미 초기화됨 (위 셀에서)
        test_preds_tta, test_logprobs_tta = vllm_tta_inference(
            vllm_engine, lora_req, sampling, vllm_processor,
            test_df, ANSWER_TOKEN_IDS, TTA_PERMUTATIONS,
        )
    except Exception as e:
        print(f'vLLM TTA failed: {e}')
        print('Falling back to HF TTA...')
        # HF fallback: reload model if needed
        if 'model_a_best' not in dir() or model_a_best is None:
            model_a_best, proc_a = load_model_for_inference(SAVE_DIR_A)
        test_preds_tta, test_logprobs_tta = tta_logprob_inference(
            model_a_best, proc_a, test_df, ANSWER_TOKEN_IDS, TTA_PERMUTATIONS,
        )
else:
    if 'model_a_best' not in dir() or model_a_best is None:
        model_a_best, proc_a = load_model_for_inference(SAVE_DIR_A)
    test_preds_tta, test_logprobs_tta = tta_logprob_inference(
        model_a_best, proc_a, test_df, ANSWER_TOKEN_IDS, TTA_PERMUTATIONS,
    )

tta_elapsed = time.time() - tta_start
print(f'\nTTA inference completed in {tta_elapsed/3600:.1f}h ({tta_elapsed/60:.0f} min)')
print(f'TTA predictions: {Counter(test_preds_tta)}')

# Single vs TTA 비교
agree = sum(a == t for a, t in zip(test_preds_a, test_preds_tta))
print(f'\nSingle vs TTA agreement: {agree}/{len(test_preds_a)} = {agree/len(test_preds_a):.4f}')
print(f'TTA changed {len(test_preds_a) - agree} predictions')

sub_tta = pd.DataFrame({'id': test_df['id'], 'answer': test_preds_tta})
sub_tta.to_csv('submission_tta.csv', index=False)
print('Saved submission_tta.csv')


## Cell 19: Memory Cleanup

Free GPU memory before optional phases.

In [ ]:
del model_a_best, proc_a
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.1f}GB")

## Cell 20: Final Submission + Sanity Checks

Generate final submission CSV and run sanity checks.

In [ ]:
# Use TTA submission as the primary submission
import shutil
shutil.copy("submission_tta.csv", "submission.csv")
print("Final submission: submission.csv (TTA-based)")

# Sanity checks
submission = pd.read_csv("submission.csv")
print(f"\nSubmission shape: {submission.shape}")
print(f"Prediction distribution: {Counter(submission['answer'].tolist())}")
print(submission.head(10))

assert len(submission) == len(test_df), f"Row count mismatch: {len(submission)} vs {len(test_df)}"
assert set(submission["answer"].unique()).issubset({"a","b","c","d"}), "Invalid answers!"
assert submission["id"].nunique() == len(submission), "Duplicate IDs!"
print("\nAll sanity checks passed.")

# Summary of all generated submissions
print("\n" + "=" * 60)
print("GENERATED SUBMISSIONS:")
print(f"  1. submission_model_a.csv  - Single model (Model A), no TTA")
print(f"  2. submission_tta.csv      - Model A + TTA (4 permutations)")
print(f"  3. submission.csv          - Final (copy of submission_tta.csv)")
print("=" * 60)

# Submission 파일 Drive 백업

In [ ]:
# ============================================================
# Submission 파일을 Google Drive에 백업
# ============================================================

if DRIVE_CHECKPOINT_DIR:
    backup_dir = os.path.dirname(DRIVE_CHECKPOINT_DIR)  # /content/drive/MyDrive
    submissions_dir = os.path.join(backup_dir, 'submissions')
    os.makedirs(submissions_dir, exist_ok=True)

    import shutil
    for fname in ['submission_model_a.csv', 'submission_tta.csv', 'submission.csv']:
        if os.path.exists(fname):
            shutil.copy(fname, os.path.join(submissions_dir, fname))
            print(f'  Backed up: {fname} -> {submissions_dir}/{fname}')
        else:
            print(f'  Skip: {fname} (not found)')

    print(f'\nAll submissions backed up to: {submissions_dir}')
else:
    print('Drive not mounted — skip backup')


---

# OPTIONAL: Phase 5 - Model B + Ensemble

**Only run if >6 hours remain after Phase 4 (TTA inference).**

Model B provides model-level diversity on top of TTA's position-debiasing.

## Optional Cell A: Train Model B

Second model with different seed, learning rate, and LoRA rank (seed=123, lr=5e-5, LoRA r=64, alpha=128).

In [ ]:
print("=" * 60)
print("OPTIONAL: TRAINING MODEL B: seed=123, lr=5e-5, LoRA r=64, alpha=128")
print("=" * 60)

model_b, processor_b = load_model_and_processor(
    MODEL_ID, IMAGE_SIZE,
    lora_r=LORA_R_B, lora_alpha=LORA_ALPHA_B,
    lora_dropout=LORA_DROPOUT, seed=SEED_B,
)

best_loss_b = train_model(
    model_b, processor_b,
    final_train_df, final_val_df,
    num_epochs=NUM_EPOCHS, lr=LR_B,
    grad_accum=GRAD_ACCUM, save_dir=SAVE_DIR_B,
    seed=SEED_B,
)
print(f"Model B best val loss: {best_loss_b:.4f}")
# Drive에 체크포인트 백업
if DRIVE_CHECKPOINT_DIR:
    import shutil
    backup_path = os.path.join(DRIVE_CHECKPOINT_DIR, "model_b")
    if os.path.exists(backup_path):
        shutil.rmtree(backup_path)
    shutil.copytree(SAVE_DIR_B, backup_path)
    print(f"Model B checkpoint backed up to Drive: {backup_path}")


## Optional Cell B: Model B Validation

Validate Model B and apply fallback guard.

In [ ]:
del model_b; gc.collect(); torch.cuda.empty_cache()
model_b_best, proc_b = load_model_for_inference(SAVE_DIR_B)

val_preds_b, val_logprobs_b = logprob_inference(
    model_b_best, proc_b, final_val_df, ANSWER_TOKEN_IDS, return_logprobs=True
)

correct_b = sum(p == g for p, g in zip(val_preds_b, val_gold))
val_accuracy_b = correct_b / len(val_gold)
print(f"Model B Val Accuracy: {correct_b}/{len(val_gold)} = {val_accuracy_b:.4f}")

# Fallback guard for Model B
if val_accuracy_b < VAL_ACCURACY_FALLBACK_THRESHOLD:
    print(f"FALLBACK GUARD: Model B accuracy ({val_accuracy_b:.4f}) < {VAL_ACCURACY_FALLBACK_THRESHOLD}")
    print("Do NOT include Model B in ensemble. Use single-model TTA submission instead.")

## Optional Cell C: Ensemble via Logprob Averaging

Average logprobs from Model A and Model B on the validation set to assess ensemble quality.

**Note**: Model A is reloaded here using `load_model_for_inference(SAVE_DIR_A)`.

In [ ]:
def logprob_ensemble(logprobs_a: List[dict], logprobs_b: List[dict]) -> List[str]:
    """Average logprobs from two models and pick the best answer."""
    predictions = []
    for lp_a, lp_b in zip(logprobs_a, logprobs_b):
        avg = {k: (lp_a[k] + lp_b[k]) / 2.0 for k in lp_a}
        predictions.append(max(avg, key=avg.get))
    return predictions

# Reload Model A for logprob extraction on val set
model_a_reloaded, proc_a_reloaded = load_model_for_inference(SAVE_DIR_A)

val_preds_a2, val_logprobs_a = logprob_inference(
    model_a_reloaded, proc_a_reloaded, final_val_df, ANSWER_TOKEN_IDS, return_logprobs=True
)

ens_val = logprob_ensemble(val_logprobs_a, val_logprobs_b)
correct_ens = sum(p == g for p, g in zip(ens_val, val_gold))
print(f"\nEnsemble Val Accuracy: {correct_ens}/{len(val_gold)} = {correct_ens/len(val_gold):.4f}")
print(f"  Model A: {val_accuracy_a:.4f}")
print(f"  Model B: {val_accuracy_b:.4f}")
print(f"  Ensemble: {correct_ens/len(val_gold):.4f}")

## Optional Cell D: Ensemble Test Inference + Submission

Run both models on the test set with logprob extraction, average logprobs, and generate ensemble submission.
Automatically selects the best submission (TTA vs ensemble) based on val accuracy.

In [ ]:
# Model A test logprobs (model_a_reloaded is still loaded from Cell C)
print("Running Model A logprob inference on test set...")
test_preds_a_ens, test_logprobs_a = logprob_inference(
    model_a_reloaded, proc_a_reloaded, test_df, ANSWER_TOKEN_IDS, return_logprobs=True
)

# Free Model A, keep Model B for test inference
del model_a_reloaded, proc_a_reloaded
gc.collect(); torch.cuda.empty_cache()

# Model B test logprobs (model_b_best is still loaded from Cell B)
print("Running Model B logprob inference on test set...")
test_preds_b_ens, test_logprobs_b = logprob_inference(
    model_b_best, proc_b, test_df, ANSWER_TOKEN_IDS, return_logprobs=True
)

# Ensemble: average logprobs from both models
test_preds_ensemble = logprob_ensemble(test_logprobs_a, test_logprobs_b)
print(f"\nEnsemble test predictions: {Counter(test_preds_ensemble)}")

# Compare with TTA predictions
agree_ens_tta = sum(e == t for e, t in zip(test_preds_ensemble, test_preds_tta))
print(f"Ensemble vs TTA agreement: {agree_ens_tta}/{len(test_preds_ensemble)} = {agree_ens_tta/len(test_preds_ensemble):.4f}")

sub_ensemble = pd.DataFrame({"id": test_df["id"], "answer": test_preds_ensemble})
sub_ensemble.to_csv("submission_ensemble.csv", index=False)
print("Saved submission_ensemble.csv")

# Decide final submission: use ensemble if it beat TTA on val, otherwise keep TTA
ens_val_acc = correct_ens / len(val_gold)
print(f"\n--- Final Decision ---")
print(f"TTA val accuracy:      {val_accuracy_a:.4f}")
print(f"Ensemble val accuracy: {ens_val_acc:.4f}")
if ens_val_acc > val_accuracy_a:
    shutil.copy("submission_ensemble.csv", "submission.csv")
    print("-> Using ENSEMBLE as final submission (submission.csv)")
else:
    print("-> Keeping TTA as final submission (submission.csv)")

# Cleanup
del model_b_best, proc_b
gc.collect(); torch.cuda.empty_cache()
print(f"\nGPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.1f}GB")

# Final summary
print("\n" + "=" * 60)
print("ALL GENERATED SUBMISSIONS:")
print(f"  1. submission_model_a.csv   - Single model (Model A), no TTA")
print(f"  2. submission_tta.csv       - Model A + TTA (4 permutations)")
print(f"  3. submission_ensemble.csv  - Model A + Model B logprob ensemble")
print(f"  4. submission.csv           - Final (best of TTA vs ensemble)")
print("=" * 60)

# Optional: 앙상블 Submission Drive 백업

In [ ]:
# ============================================================
# 앙상블 Submission 파일을 Google Drive에 백업
# ============================================================

if DRIVE_CHECKPOINT_DIR:
    backup_dir = os.path.dirname(DRIVE_CHECKPOINT_DIR)
    submissions_dir = os.path.join(backup_dir, 'submissions')
    os.makedirs(submissions_dir, exist_ok=True)

    import shutil
    for fname in ['submission_ensemble.csv', 'submission.csv']:
        if os.path.exists(fname):
            shutil.copy(fname, os.path.join(submissions_dir, fname))
            print(f'  Backed up: {fname} -> {submissions_dir}/{fname}')

    print(f'\nEnsemble submissions backed up to: {submissions_dir}')
else:
    print('Drive not mounted — skip backup')
